# IVECO Statistics - Modular Main Pipeline

Notebook finale con una cella di controllo per ogni sheet Excel.

In [ ]:
from run_local_sample import (
    DEFAULT_FORMAT,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_SAMPLE_PATH,
    build_spark,
    build_sheet_pivot,
    export_excel_outputs,
    read_sample,
    validate_config_columns,
)
from engine_cleaning import add_legacy_preparation_features, clean_spark_column_names, keep_latest_record_per_vin
from engine_config import get_default_sheet_ids
from engine_loader import extract_metadata, import_fat_tables_3
from engine_utils import log_step, report_dim, report_vin

In [ ]:
def parse_config_text(value):
    if isinstance(value, (set, list, tuple)):
        return {int(v) for v in value}
    parts = str(value).replace(';', ',').replace(' ', ',').split(',')
    config_values = {int(part.strip()) for part in parts if part.strip()}
    if not config_values:
        raise ValueError("Config vuota: inserisci almeno un id_config, es. 399 oppure 399,400")
    return config_values


DEFAULT_CONFIG = {399}
DEFAULT_INPUT_MODE = "sample"


def read_input_data(spark, input_mode, sample_path, input_format, config):
    if input_mode == "fat_table":
        return import_fat_tables_3(spark, config)
    return read_sample(spark, sample_path, input_format)


default_config_text = ','.join(map(str, sorted(DEFAULT_CONFIG)))

try:
    dbutils
    try:
        dbutils.widgets.get("config")
    except Exception:
        dbutils.widgets.text("config", default_config_text, "Config fat table")
    try:
        dbutils.widgets.get("input_mode")
    except Exception:
        dbutils.widgets.dropdown("input_mode", "fat_table", ["fat_table", "sample"], "Input mode")
    try:
        dbutils.widgets.get("output_dir")
    except Exception:
        dbutils.widgets.text("output_dir", "Excel_statistics", "Output dir")

    input_mode = dbutils.widgets.get("input_mode")
    config_text = dbutils.widgets.get("config")
    output_dir = dbutils.widgets.get("output_dir")
except NameError:
    input_mode = DEFAULT_INPUT_MODE
    config_text = default_config_text
    output_dir = DEFAULT_OUTPUT_DIR

config = parse_config_text(config_text)
sample_path = DEFAULT_SAMPLE_PATH
input_format = DEFAULT_FORMAT
sheet_ids = get_default_sheet_ids()
keep_latest_per_vin = True
export_excel = True

log_step(f"Input mode: {input_mode}")
log_step(f"Config selezionate: {sorted(config)}")
log_step(f"Output dir: {output_dir}")
log_step(f"Sample path: {sample_path}")
log_step(f"Sheet configurati: {sheet_ids}")

## 1. Lettura input

In [ ]:
try:
    spark
except NameError:
    spark = build_spark("iveco-main-pipeline-modular")

df_raw = read_input_data(spark, input_mode, sample_path, input_format, config)
report_dim(df_raw, "RAW input")
report_vin(df_raw)

## 2. Cleaning e ultimo record per VIN

In [ ]:
df_clean = clean_spark_column_names(df_raw)
report_dim(df_clean, "CLEAN sample")

if keep_latest_per_vin:
    df_clean = keep_latest_record_per_vin(df_clean)
    report_dim(df_clean, "LATEST VIN sample")
    report_vin(df_clean)

df_clean = add_legacy_preparation_features(df_clean)
report_dim(df_clean, "PREP FEATURES sample")

## 3. Metadata e validazione configurazione

In [ ]:
p_type, p_group, p_series = extract_metadata(df_clean)
log_step(f"Metadata rilevati: type={p_type}, group={p_group}, series={p_series}")

validation = validate_config_columns(df_clean, p_series, p_group, sheet_ids)

## 4. Helper preview sheet

In [ ]:
sheet_outputs_by_id = {}

def preview_sheet(sheet_id):
    validation_entry = validation[sheet_id]
    df_sheet, sheet_name = build_sheet_pivot(df_clean, sheet_id, validation_entry)

    log_step(
        f"Preview {sheet_id} - {sheet_name}: "
        f"presenti={len(validation_entry['present'])}, "
        f"mancanti={len(validation_entry['missing'])}"
    )

    if validation_entry["missing"]:
        print(f"Colonne mancanti per {sheet_id}: {validation_entry['missing']}")

    if df_sheet is None or df_sheet.empty:
        print(f"Sheet saltato: {sheet_id} - {sheet_name}")
        return None

    sheet_outputs_by_id[sheet_id] = {
        "sheet_id": sheet_id,
        "sheet_name": sheet_name,
        "dataframe": df_sheet,
        "validation": validation_entry,
    }

    try:
        display(df_sheet)
    except NameError:
        print(df_sheet.to_string(index=False))

    return df_sheet

# Sheet Excel

## Sheet 5) CATALYST INFO

In [ ]:
df_sheet_catalyst_info = preview_sheet("catalyst_info")

## Sheet 7) MAXIMUM VALUES

In [ ]:
df_sheet_max_values = preview_sheet("max_values")

## Sheet 1a) Engine Torque / Engine Speed

In [ ]:
df_sheet_1a = preview_sheet("1a")

## Sheet 1b) Engine Torque / Vehicle Speed

In [ ]:
df_sheet_1b = preview_sheet("1b")

## Sheet 1c) Engine Revolutions

In [ ]:
df_sheet_1c = preview_sheet("1c")

## Sheet 1d) Turbo Overspeed

In [ ]:
df_sheet_1d = preview_sheet("1d")

## Sheet 2a) Oil Pressure

In [ ]:
df_sheet_2a = preview_sheet("2a")

## Sheet 2b) Oil Sump Pressure

In [ ]:
df_sheet_2b = preview_sheet("2b")

## Sheet 2c) Oil Temperature

In [ ]:
df_sheet_2c = preview_sheet("2c")

## Sheet 3a) Coolant Temperature

In [ ]:
df_sheet_3a = preview_sheet("3a")

## Sheet 3a.1) Coolant Pressure

In [ ]:
df_sheet_3a_1 = preview_sheet("3a_1")

## Sheet 3b) Fuel Temperature

In [ ]:
df_sheet_3b = preview_sheet("3b")

## Sheet 3c) Intake Air Temperature

In [ ]:
df_sheet_3c = preview_sheet("3c")

## Sheet 3c.1) Fuel Pre-Filter Pressure

In [ ]:
df_sheet_3c_1 = preview_sheet("3c_1")

## Sheet 3d) Intake Air Pressure

In [ ]:
df_sheet_3d = preview_sheet("3d")

## Sheet 3e) Flap Actuator Position

In [ ]:
df_sheet_3e = preview_sheet("3e")

## Sheet 3f) EGR Actuator Position

In [ ]:
df_sheet_3f = preview_sheet("3f")

## Sheet 4a.1) Catalyst Efficiency PEMS

In [ ]:
df_sheet_4a_1 = preview_sheet("4a_1")

## Sheet 4a) Catalyst Efficiency

In [ ]:
df_sheet_4a = preview_sheet("4a")

## Sheet 4b) Catalyst Efficiency vs Engine Speed

In [ ]:
df_sheet_4b = preview_sheet("4b")

## Sheet 4c) HC Accumulated in SCR Catalyst

In [ ]:
df_sheet_4c = preview_sheet("4c")

## Sheet 4d) NH3 Concentration

In [ ]:
df_sheet_4d = preview_sheet("4d")

## Sheet 4e) AdBlue Pressure Pump

In [ ]:
df_sheet_4e = preview_sheet("4e")

## Sheet 4f) Upstream Temperature

In [ ]:
df_sheet_4f = preview_sheet("4f")

## Sheet 4h) SCR Downstream Temperature

In [ ]:
df_sheet_4h = preview_sheet("4h")

## Sheet 5a) DPF Upstream Temperature

In [ ]:
df_sheet_5a_dpf = preview_sheet("5a_dpf")

## Sheet 5b) Regeneration Strategies

In [ ]:
df_sheet_5b = preview_sheet("5b")

## Sheet 5c) DPF Differential Pressure

In [ ]:
df_sheet_5c = preview_sheet("5c")

## Sheet 5d) Soot Mass Estimated

In [ ]:
df_sheet_5d = preview_sheet("5d")

## Sheet aggiuntivi da configurazione

In [ ]:
for sheet_id in sheet_ids:
    if sheet_id not in sheet_outputs_by_id:
        preview_sheet(sheet_id)

## Export Excel

In [ ]:
if export_excel:
    ordered_sheet_outputs = [
        sheet_outputs_by_id[sheet_id]
        for sheet_id in sheet_ids
        if sheet_id in sheet_outputs_by_id
    ]
    excel_path = export_excel_outputs(ordered_sheet_outputs, output_dir)
    log_step(f"Excel generato: {excel_path}")
else:
    log_step("Export Excel disabilitato")

In [ ]:
log_step("Main pipeline modulare completata")